# Chapter 3: Templates with Jinja2 🎨

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='2. routing_and_url_building.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 2: Routing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../2.%20working_with_data/4.%20handling_forms_and_user_input.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 4: Forms →</a>
</div>

---

**What you'll learn in this chapter:**
- What a template engine is and *why* you need one
- Jinja2 syntax: variables `{{ }}`, statements `{% %}`, comments `{# #}`
- Filters, control flow (if/for), template inheritance, and macros
- How Flask's `render_template()` / `render_template_string()` work
- Best practices for static files with `url_for`

> ❓ **Why use a framework's routing instead of `if url == '/': ...`?** A routing system matches URLs against patterns (including variables like `<int:id>`), dispatches to the right function, and handles edge cases like trailing slashes and HTTP method checks — without any boilerplate from you.

---

## 🎯 The Big Picture

Returning raw strings from Flask routes works for "Hello World", but real web pages have structure, styling, and dynamic data. **Templates** are HTML files with special slots where your Python data gets inserted. **Jinja2** is the engine that fills in those slots.

```
Python data  ──►  Jinja2 engine  ──►  Rendered HTML  ──►  Browser
{"name":"Alice"}    (fills slots)     <h1>Hello Alice</h1>
```

Think of it like a **mail-merge**: the letter template is fixed, but each recipient gets their own name and address filled in automatically.


---

## 🧠 Core Concepts: The "Why"

### What is a template engine?

> 💡 **Analogy:** A template is like a form letter — the structure and words are fixed, but the *name* and *address* slots are filled in for each recipient.

Without templates you'd build HTML by string-concatenation in Python — ugly, error-prone, and impossible to maintain.

---

### Separation of Concerns

| Layer | Responsibility | File type |
|-------|---------------|-----------|
| Python / Flask | Business logic, data fetching | `.py` |
| Jinja2 templates | Presentation, layout | `.html` |
| CSS | Visual styling | `.css` |

Keeping these layers separate makes each one easier to read, test, and change.

---

### How Jinja2 fits into Flask

Flask ships with Jinja2 built-in. The key function is **`render_template()`**, which:

1. Looks for a file in your app's `templates/` folder
2. Parses it with Jinja2
3. Substitutes your Python variables into the placeholders
4. Returns the finished HTML string to the browser

```
your_project/
├── app.py
└── templates/         ← Flask looks here by default
    ├── base.html
    ├── index.html
    └── about.html
```

---

### Three Jinja2 syntax types

| Syntax | Purpose | Example |
|--------|---------|---------|
| `{{ expression }}` | **Output** a value | `{{ user.name }}` |
| `{% statement %}` | **Control flow** (if, for, block…) | `{% if logged_in %}` |
| `{# comment #}` | **Comment** — never rendered to HTML | `{# TODO: fix this #}` |


---

## ⚡ Syntax & First Usage

### `render_template()` and `render_template_string()`

- **`render_template('index.html', name='Alice')`** — loads `templates/index.html` from disk.
- **`render_template_string('<h1>Hello {{ name }}</h1>', name='Alice')`** — renders a template from a Python string (great for demos and testing).

Since we can't write to the filesystem during notebook execution, we'll use `render_template_string()` and Jinja2's `Environment` directly throughout this chapter — the behaviour is identical to `render_template()`.

Let's start with the simplest possible example: variable substitution.


In [ ]:
from jinja2 import Environment

env = Environment()
template = env.from_string("Hello, {{ name }}! You have {{ count }} messages.")
result = template.render(name="Alice", count=5)
print(result)


### The `templates/` folder

When you call `render_template('greeting.html', name='Alice', count=5)` in a real Flask app, Flask looks for `templates/greeting.html` relative to your app file.

```python
# app.py
from flask import Flask, render_template

app = Flask(__name__)

@app.route('/hello/<name>')
def hello(name):
    return render_template('greeting.html', name=name, count=5)
```

```html
<!-- templates/greeting.html -->
<!DOCTYPE html>
<html>
<body>
  <p>Hello, {{ name }}! You have {{ count }} messages.</p>
</body>
</html>
```

Flask passes every keyword argument you give `render_template()` into the template context — those become the variables available inside the template.

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.route('/')` is shorthand for `index = app.route('/')(index)` — it registers your view function with Flask's URL map without any explicit call.

In [ ]:
from flask import Flask, render_template_string

app = Flask(__name__)

html_template = (
    "<!DOCTYPE html>\n"
    "<html>\n"
    "<head><title>{{ page_title }}</title></head>\n"
    "<body>\n"
    "  <h1>Welcome, {{ username }}!</h1>\n"
    "  <p>Your role is: <strong>{{ role }}</strong></p>\n"
    "  {% if is_admin %}\n"
    "  <p style=\"color:red;\">\U0001f511 Admin panel is visible to you.</p>\n"
    "  {% endif %}\n"
    "</body>\n"
    "</html>"
)

with app.test_request_context():
    output = render_template_string(
        html_template,
        page_title="Dashboard",
        username="Alice",
        role="admin",
        is_admin=True,
    )
    print(output)


---

## 🔬 Deep Dive

### Variables and Filters

**Filters** transform a variable's value before it's output. You apply them with a pipe character `|`:

```
{{ name | upper }}          → ALICE
{{ bio  | truncate(30) }}   → "She writes code and drink..."
{{ tags | join(', ') }}     → "python, flask, web"
```

Think of filters as small helper functions that live inside the template. Here are the most common ones:

| Filter | What it does | Example |
|--------|-------------|---------|
| `upper` | Uppercase | `{{ 'hello' \| upper }}` → `HELLO` |
| `lower` | Lowercase | `{{ 'HELLO' \| lower }}` → `hello` |
| `title` | Title Case | `{{ 'hello world' \| title }}` → `Hello World` |
| `truncate(n)` | Truncate to n chars | `{{ long_text \| truncate(20) }}` |
| `length` | Count items / chars | `{{ items \| length }}` |
| `default('N/A')` | Fallback if None/empty | `{{ value \| default('N/A') }}` |
| `join(', ')` | Join a list | `{{ tags \| join(', ') }}` |
| `safe` | Mark as safe HTML (use carefully) | `{{ html_content \| safe }}` |
| `escape` | HTML-escape (default) | `{{ user_input \| escape }}` |
| `int` | Convert to integer | `{{ '42' \| int }}` |
| `float` | Convert to float | `{{ '3.14' \| float }}` |
| `round` | Round a number | `{{ 3.7 \| round }}` |
| `replace(a, b)` | String replace | `{{ text \| replace('bad', 'good') }}` |
| `trim` | Strip whitespace | `{{ '  hi  ' \| trim }}` |


In [ ]:
from jinja2 import Environment

env = Environment()

demos = {
    "upper":     "{{ 'hello world' | upper }}",
    "lower":     "{{ 'HELLO WORLD' | lower }}",
    "title":     "{{ 'hello world' | title }}",
    "truncate":  "{{ 'She writes code and drinks coffee' | truncate(20) }}",
    "length":    "{{ ['a', 'b', 'c'] | length }}",
    "default":   "{{ missing_var | default('N/A') }}",
    "join":      "{{ ['python', 'flask', 'web'] | join(', ') }}",
    "replace":   "{{ 'I hate bugs' | replace('hate', 'love') }}",
    "trim":      "{{ '   spaced   ' | trim }}",
    "int":       "{{ '42' | int + 8 }}",
    "round":     "{{ 3.7 | round }}",
}

for name, template_str in demos.items():
    tmpl = env.from_string(template_str)
    result = tmpl.render()
    print(f"  {name:10s}: {result}")


---

### Control Flow: `if` / `elif` / `else`

Jinja2 if-statements look almost like Python, but they're wrapped in `{% %}` tags and need a closing `{% endif %}`:

```html
{% if score >= 90 %}
  <span class="grade-a">A</span>
{% elif score >= 80 %}
  <span class="grade-b">B</span>
{% else %}
  <span class="grade-c">C or below</span>
{% endif %}
```

> 💡 **Tip:** There's no indentation requirement in Jinja2 — the `{% endif %}` tag is what closes the block.


In [ ]:
from jinja2 import Environment

env = Environment()

grade_template = env.from_string(
    "{%- if score >= 90 -%}\n"
    "  Grade: A -- Excellent!\n"
    "{%- elif score >= 80 -%}\n"
    "  Grade: B -- Good job.\n"
    "{%- elif score >= 70 -%}\n"
    "  Grade: C -- Passing.\n"
    "{%- else -%}\n"
    "  Grade: F -- Please review the material.\n"
    "{%- endif -%}"
)

for score in [95, 83, 72, 55]:
    print(f"Score {score}: {grade_template.render(score=score)}")


### Control Flow: `for` Loops

Jinja2 for loops iterate over any Python iterable. Inside the loop, a special `loop` object gives you useful metadata:

| Variable | Meaning |
|----------|---------|
| `loop.index` | Current iteration (1-based) |
| `loop.index0` | Current iteration (0-based) |
| `loop.first` | `True` on first iteration |
| `loop.last` | `True` on last iteration |
| `loop.length` | Total number of items |

```html
{% for item in items %}
  <li {% if loop.first %}class="first"{% endif %}>
    {{ loop.index }}. {{ item }}
  </li>
{% else %}
  <li>No items found.</li>
{% endfor %}
```

The `{% else %}` block runs when the list is empty — handy for "no results" messages.


In [ ]:
from jinja2 import Environment

env = Environment()

menu_template = env.from_string(
    "\nNavigation menu ({{ items | length }} items):\n"
    "{%- for item in items %}\n"
    "  {{ loop.index }}. {{ item.label }}"
    "{%- if loop.first %} <- START{% endif %}"
    "{%- if loop.last %}  <- END{% endif %}\n"
    "{%- else %}\n"
    "  (empty menu)\n"
    "{%- endfor %}\n"
)

menu_items = [
    {"label": "Home"},
    {"label": "Blog"},
    {"label": "About"},
    {"label": "Contact"},
]

print(menu_template.render(items=menu_items))
print("---")
print(menu_template.render(items=[]))  # empty case


---

### 🏗️ Template Inheritance — THE Most Important Concept

> 💡 **Analogy:** `base.html` is the master cookie cutter — same shape every time. Child templates are different *flavors* — same structure, different content.

Without template inheritance, every HTML page would duplicate the `<head>`, navbar, and footer. Change the navbar? Edit 50 files. With inheritance, you change `base.html` once and every page updates automatically.

**How it works:**

1. `base.html` defines the full page skeleton with **`{% block name %}...{% endblock %}`** placeholders
2. Child templates start with **`{% extends 'base.html' %}`** and override only the blocks they need
3. Any block left un-overridden uses the default content from `base.html`

```
base.html          child.html
+----------+       {% extends 'base.html' %}
|  <head>  |
|  navbar  |  <-- Same (inherited)
| {% block |
|  content |  <-- Overridden by child
| %}...    |
|  footer  |  <-- Same (inherited)
+----------+
```


In [ ]:
# Let's define base.html as a Python string so we can see it clearly
base_html = (
    "<!DOCTYPE html>\n"
    "<html lang=\"en\">\n"
    "<head>\n"
    "  <meta charset=\"UTF-8\">\n"
    "  <title>{% block title %}My Flask Site{% endblock %}</title>\n"
    "  {% block head_extras %}{% endblock %}\n"
    "</head>\n"
    "<body>\n"
    "  <!-- Navbar: defined once, appears on every page -->\n"
    "  <nav>\n"
    "    <a href=\"/\">Home</a> |\n"
    "    <a href=\"/blog\">Blog</a> |\n"
    "    <a href=\"/about\">About</a>\n"
    "  </nav>\n"
    "\n"
    "  <!-- Every child template fills in this block -->\n"
    "  <main>\n"
    "    {% block content %}\n"
    "      <p>Default content -- override me in a child template.</p>\n"
    "    {% endblock %}\n"
    "  </main>\n"
    "\n"
    "  <!-- Footer: defined once -->\n"
    "  <footer>\n"
    "    <p>&copy; 2024 My Flask Site</p>\n"
    "    {% block scripts %}{% endblock %}\n"
    "  </footer>\n"
    "</body>\n"
    "</html>"
)

print(base_html)


In [ ]:
# A child template only defines what is DIFFERENT from base.html
home_html = (
    "{% extends \"base.html\" %}\n\n"
    "{% block title %}Home -- My Flask Site{% endblock %}\n\n"
    "{% block content %}\n"
    "  <h1>Welcome to My Flask Site!</h1>\n"
    "  <p>Hello, {{ username }}! Great to see you here.</p>\n"
    "  <ul>\n"
    "  {% for post in recent_posts %}\n"
    "    <li><a href=\"/post/{{ post.id }}\">{{ post.title }}</a></li>\n"
    "  {% endfor %}\n"
    "  </ul>\n"
    "{% endblock %}\n\n"
    "{% block scripts %}\n"
    "  <script>console.log('Home page loaded');</script>\n"
    "{% endblock %}"
)

print(home_html)


In [ ]:
from jinja2 import Environment, DictLoader

BASE = (
    "<!DOCTYPE html>\n<html>\n"
    "<head><title>{% block title %}Site{% endblock %}</title></head>\n"
    "<body>\n"
    "  <nav>[Home] [Blog] [About]</nav>\n"
    "  <main>{% block content %}(empty){% endblock %}</main>\n"
    "  <footer>© 2024 My Flask Blog{% block scripts %}{% endblock %}</footer>\n"
    "</body>\n</html>"
)

HOME = (
    '{% extends "base.html" %}\n'
    "{% block title %}Home{% endblock %}\n"
    "{% block content %}\n"
    "  <h1>Welcome, {{ username }}!</h1>\n"
    "  <p>{{ tagline }}</p>\n"
    "{% endblock %}"
)

ABOUT = (
    '{% extends "base.html" %}\n'
    "{% block title %}About Us{% endblock %}\n"
    "{% block content %}\n"
    "  <h1>About</h1>\n"
    "  <p>{{ description }}</p>\n"
    "{% endblock %}"
)

POST = (
    '{% extends "base.html" %}\n'
    "{% block title %}{{ post.title }}{% endblock %}\n"
    "{% block content %}\n"
    "  <h1>{{ post.title }}</h1>\n"
    "  <p>By {{ post.author }} on {{ post.date }}</p>\n"
    "  <article>{{ post.body }}</article>\n"
    "{% endblock %}\n"
    "{% block scripts %}<script>console.log('Post page');</script>{% endblock %}"
)

env = Environment(loader=DictLoader({
    "base.html": BASE,
    "home.html": HOME,
    "about.html": ABOUT,
    "post.html": POST,
}))

print("=== HOME PAGE ===")
print(env.get_template("home.html").render(
    username="Alice", tagline="Building the web, one route at a time."
))

print("\n=== ABOUT PAGE ===")
print(env.get_template("about.html").render(
    description="A learning project built with Flask and Jinja2."
))

print("\n=== POST PAGE ===")
print(env.get_template("post.html").render(post={
    "title": "Getting Started with Flask",
    "author": "Alice",
    "date": "2024-01-15",
    "body": "Flask is a lightweight Python web framework...",
}))


---

### 🔧 Macros — Reusable Template Components

> 💡 **Macros are like functions for HTML.** Define a chunk of markup once, call it many times with different arguments.

Without macros, you'd copy-paste the same `<input>` markup for every form field. With macros, you define it once and call it like a function.

```html
{% macro input_field(name, label, type='text', required=False) %}
  <div class="form-group">
    <label for="{{ name }}">{{ label }}</label>
    <input type="{{ type }}" id="{{ name }}" name="{{ name }}"
           {% if required %}required{% endif %}>
  </div>
{% endmacro %}

{# Use it like a function #}
{{ input_field('email', 'Email Address', type='email', required=True) }}
{{ input_field('username', 'Username', required=True) }}
{{ input_field('bio', 'Bio') }}
```


In [ ]:
from jinja2 import Environment

env = Environment()

macro_src = (
    "{%- macro input_field(name, label, type='text', required=False) -%}\n"
    "<div class=\"form-group\">\n"
    "  <label for=\"{{ name }}\">{{ label }}{% if required %} *{% endif %}</label>\n"
    "  <input type=\"{{ type }}\" id=\"{{ name }}\" name=\"{{ name }}\"{{ ' required' if required else '' }}>\n"
    "</div>\n"
    "{%- endmacro -%}\n\n"
    "{%- macro alert(message, level='info') -%}\n"
    "<div class=\"alert alert-{{ level }}\" role=\"alert\">\n"
    "  {% if level == 'danger' %}⚠️{% elif level == 'success' %}✅{% else %}ℹ️{% endif %}\n"
    "  {{ message }}\n"
    "</div>\n"
    "{%- endmacro -%}\n\n"
    "=== Registration Form Fields ===\n"
    "{{ input_field('username', 'Username', required=True) }}\n"
    "{{ input_field('email', 'Email Address', type='email', required=True) }}\n"
    "{{ input_field('password', 'Password', type='password', required=True) }}\n"
    "{{ input_field('website', 'Website (optional)') }}\n\n"
    "=== Alert Messages ===\n"
    "{{ alert('Profile saved successfully!', 'success') }}\n"
    "{{ alert('Your session will expire in 5 minutes.', 'info') }}\n"
    "{{ alert('Incorrect password. Please try again.', 'danger') }}\n"
)

print(env.from_string(macro_src).render())


---

### 📁 Static Files with `url_for`

**Never hardcode paths to CSS, JS, or images.** Use `url_for('static', filename='...')` instead.

| Hardcoded (fragile) | `url_for` (robust) |
|--------------------|--------------------|
| `<link href="/static/css/style.css">` | `<link href="{{ url_for('static', filename='css/style.css') }}">` |
| Breaks if app is mounted at a sub-path | Always generates the correct URL |
| No fingerprinting support | Works with cache headers |

Your static files go in a `static/` folder alongside `templates/`:

```
your_project/
├── app.py
├── templates/
│   └── base.html
└── static/
    ├── css/
    │   └── style.css
    ├── js/
    │   └── app.js
    └── img/
        └── logo.png
```

In `base.html`:
```html
<link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
<img src="{{ url_for('static', filename='img/logo.png') }}" alt="Logo">
<script src="{{ url_for('static', filename='js/app.js') }}"></script>
```


In [ ]:
from flask import Flask, url_for

app = Flask(__name__)

with app.test_request_context():
    css_url  = url_for('static', filename='css/style.css')
    js_url   = url_for('static', filename='js/app.js')
    logo_url = url_for('static', filename='img/logo.png')
    favicon  = url_for('static', filename='favicon.ico')

    print("Generated static URLs:")
    print(f"  CSS    : {css_url}")
    print(f"  JS     : {js_url}")
    print(f"  Logo   : {logo_url}")
    print(f"  Favicon: {favicon}")
    print()
    print("In your base.html template:")
    print("  <link rel=\"stylesheet\" href=\"{{ url_for('static', filename='css/style.css') }}\">")
    print("  <script src=\"{{ url_for('static', filename='js/app.js') }}\"></script>")


---

### ⚖️ Comparison 1: Raw String vs `render_template_string`

Let's see the same "user profile" page built two ways.


In [ ]:
from flask import Flask, render_template_string

app = Flask(__name__)
user = {"name": "Alice", "role": "Admin", "posts": 42, "joined": "2023-01-15"}

# Way 1: Building HTML by string concatenation
def profile_raw(u):
    html = "<!DOCTYPE html><html><body>"
    html += "<h1>Profile: " + u["name"] + "</h1>"
    html += "<p>Role: <b>" + u["role"] + "</b></p>"
    html += "<p>Posts: " + str(u["posts"]) + "</p>"
    if u["role"] == "Admin":
        html += "<p style='color:red'>Admin privileges active</p>"
    html += "<p>Joined: " + u["joined"] + "</p>"
    html += "</body></html>"
    return html

# Way 2: Using render_template_string
profile_template = (
    "<!DOCTYPE html>\n<html>\n<body>\n"
    "  <h1>Profile: {{ user.name }}</h1>\n"
    "  <p>Role: <b>{{ user.role }}</b></p>\n"
    "  <p>Posts: {{ user.posts }}</p>\n"
    "  {% if user.role == 'Admin' %}\n"
    "  <p style=\"color:red\">Admin privileges active</p>\n"
    "  {% endif %}\n"
    "  <p>Joined: {{ user.joined }}</p>\n"
    "</body>\n</html>"
)

with app.test_request_context():
    rendered = render_template_string(profile_template, user=user)

print("RAW STRING approach (messy concatenation):")
print(profile_raw(user))
print()
print("TEMPLATE approach (same output, much cleaner):")
print(rendered)


The output is identical — but the template version is:
- **Readable** by both Python devs and HTML/CSS designers
- **Maintainable** — no string escaping nightmares
- **Safe** — Jinja2 auto-escapes `<`, `>`, `&` to prevent XSS
- **Separable** — a designer can edit the `.html` file without touching Python

---

### ⚖️ Comparison 2: Copy-Paste HTML vs Template Inheritance

Imagine you have 10 pages and each one copies the navbar manually.


In [ ]:
# WITHOUT inheritance — navbar duplicated in every template
print("WITHOUT inheritance:")
print("  Each of 10 pages contains the full navbar block (~8 lines).")
print("  Change the navbar? -> Edit 10 files manually (and probably miss one).")
print(f"  Total lines for 10 pages: ~{20 * 10} (lots of duplication)")
print()

# WITH inheritance — navbar defined ONCE in base.html
print("WITH inheritance:")
print("  base.html defines navbar once (~15 lines total).")
print("  Each child template is tiny -- only unique content (~8 lines).")
print(f"  Total lines for 10 pages: ~{15 + 8 * 10} (base + 10 children)")
print(f"  Lines saved: ~{20*10 - (15 + 8*10)}")
print()

# Show what a child looks like
child_example = (
    '{% extends "base.html" %}\n'
    "{% block title %}Home{% endblock %}\n"
    "{% block content %}\n"
    "  <h1>Welcome!</h1>\n"
    "  <p>This is ALL that's unique to this page.</p>\n"
    "{% endblock %}"
)
print("What each child template looks like:")
print(child_example)
print()
print("Change the navbar once in base.html -> all 10 pages update automatically.")


---

## 🧪 What If? Experimentation

These three scenarios explore common pitfalls you'll hit in real projects.


### Q1: What if a variable passed to the template is `None`?

This is one of the most common bugs. A database query returns `None` for an optional field, and suddenly your page shows the word "None" to users.


In [ ]:
from jinja2 import Environment

env = Environment()

# Simulate database records where some fields are missing
users = [
    {"name": "Alice",   "bio": "Flask developer",  "website": "https://alice.dev"},
    {"name": "Bob",     "bio": None,                "website": None},
    {"name": "Charlie", "bio": "",                  "website": "https://charlie.io"},
]

# WITHOUT default filter
tpl_bad = env.from_string(
    "{%- for user in users %}\n"
    "{{ user.name }}: bio={{ user.bio }}, site={{ user.website }}\n"
    "{%- endfor %}"
)
print("Without | default filter:")
print(tpl_bad.render(users=users))

# WITH default filter (catches None)
tpl_good = env.from_string(
    "{%- for user in users %}\n"
    "{{ user.name }}: bio={{ user.bio | default('No bio yet') }}, "
    "site={{ user.website | default('N/A') }}\n"
    "{%- endfor %}"
)
print("\nWith | default('...') filter:")
print(tpl_good.render(users=users))

# default(value, true) also catches empty strings
tpl_best = env.from_string(
    "{%- for user in users %}\n"
    "{{ user.name }}: bio={{ user.bio | default('No bio yet', true) }}\n"
    "{%- endfor %}"
)
print("\nWith | default('...', true) -- also catches empty strings:")
print(tpl_best.render(users=users))


---

### Q2: What if you use `| safe` on user-supplied input?

**This is a critical security issue.** The `| safe` filter tells Jinja2 "I trust this string — don't escape it." That's fine for HTML you generate yourself, but dangerous for user input.

> ⚠️ **XSS (Cross-Site Scripting):** An attacker submits `<script>alert('hacked')</script>` as their username. If you render it with `| safe`, that script executes in every visitor's browser — they can steal cookies, hijack sessions, and more.

Jinja2 **auto-escapes** by default for `.html` templates in Flask, converting `<` to `&lt;`. This is your built-in XSS protection. The `| safe` filter bypasses it.


In [ ]:
from jinja2 import Environment

env_safe   = Environment(autoescape=True)   # Flask default for .html
env_unsafe = Environment(autoescape=False)

malicious_input = '<script>document.cookie="stolen="+document.cookie</script>'
template_str    = "Hello, {{ username }}!"

# With autoescaping ON (Flask default)
safe_result = env_safe.from_string(template_str).render(username=malicious_input)
print("With autoescaping ON (Flask default for .html):")
print(f"  {safe_result}")
print("  -> The <script> tag is escaped to &lt;script&gt; -- harmless text")
print()

# What | safe would do -- dangerous!
unsafe_tpl  = "Hello, {{ username | safe }}!"
unsafe_result = env_unsafe.from_string(unsafe_tpl).render(username=malicious_input)
print("With | safe on user input (DANGEROUS):")
print(f"  {unsafe_result}")
print("  -> The raw <script> tag appears in HTML -- executes in the browser!")
print()
print("RULE: Only use | safe on HTML you generated yourself.")
print("      For user-generated rich text, sanitize with bleach/nh3 FIRST,")
print("      then apply | safe.")


---

### Q3: What if you put templates in the wrong folder?

Flask looks for templates in a `templates/` subdirectory. If the file isn't there, you get a `TemplateNotFound` exception — one of the most common Flask errors for beginners.


In [ ]:
from flask import Flask, render_template
from jinja2 import TemplateNotFound

app = Flask(__name__)

with app.test_request_context():
    try:
        render_template('nonexistent_page.html')
    except TemplateNotFound as e:
        print(f"TemplateNotFound: {e}")
        print()
        print("Common causes:")
        print("  1. Typo in filename:  render_template('indx.html') not 'index.html'")
        print("  2. Wrong folder:      file is in 'template/' not 'templates/'")
        print("  3. Wrong location:    file is in project root, not templates/")
        print("  4. Blueprint path:    forgot template_folder='templates' in Blueprint()")
        print()
        print("Fix checklist:")
        print("  [x] File is at <app_root>/templates/<filename>.html")
        print("  [x] Filename matches exactly (case-sensitive on Linux/Mac)")
        print("  [x] app = Flask(__name__)  -- __name__ tells Flask where to look")
        print("  [x] Blueprint: Blueprint('name', __name__, template_folder='templates')")


---

## 🚀 Real-World Project Link

Everything you've learned in this chapter directly powers our **Flask Blog** project:

```
flask_blog/
└── templates/
    ├── base.html          ← navbar, footer, CSS links — defined ONCE
    ├── post_list.html     ← {% extends "base.html" %} — just the post list
    ├── post_detail.html   ← {% extends "base.html" %} — just the post content
    ├── login.html         ← {% extends "base.html" %} — just the login form
    └── register.html      ← {% extends "base.html" %} — just the signup form
```

**What this means in practice:**

- 🎨 Our designer updates the navbar in `base.html` once → all 5 pages update instantly
- 📱 We add a mobile menu to `base.html` → every page becomes mobile-friendly
- 🔒 We add a login banner to `base.html` → it appears everywhere automatically
- ✏️  Each child template is short (20–40 lines) — only unique content

**In the next chapter** (Forms & User Input), we'll build the login form using the `base.html` we've been designing, and you'll see how `{% extends %}` and `{% block %}` make it trivial to write.


---

## 📋 Chapter Summary & Cheat Sheet

You've covered the full Jinja2 toolkit that real Flask apps use every day. Here's your quick-reference guide.


In [ ]:
cheat_sheet = (
    '=' * 58 + '\n'
    + 'JINJA2 + FLASK TEMPLATES -- COMPLETE CHEAT SHEET\n'
    + '=' * 58 + '\n\n'
    + '1. JINJA2 SYNTAX\n'
    + '-' * 58 + '\n'
    + '{{ variable }}                    Output a value\n'
    + '{{ user.name }}                   Attribute access\n'
    + '{{ items[0] }}                    Index access\n'
    + '{% if x %}...{% endif %}          Conditional\n'
    + '{% if x %}...{% elif y %}...{% else %}...{% endif %}\n'
    + '{% for item in items %}...{% endfor %}   Loop\n'
    + '{# comment -- never rendered #}   Template comment\n\n'
    + '2. COMMON FILTERS\n'
    + '-' * 58 + '\n'
    + '{{ text | upper }}                UPPERCASE\n'
    + '{{ text | lower }}                lowercase\n'
    + '{{ text | title }}                Title Case\n'
    + '{{ text | truncate(30) }}         Truncate to 30 chars\n'
    + '{{ text | trim }}                 Strip whitespace\n'
    + '{{ text | replace(\"a\",\"b\") }}     String replace\n'
    + '{{ value | default(\"N/A\") }}      Fallback if None\n'
    + '{{ value | default(\"N/A\", true)}} Fallback if None or \"\"\n'
    + '{{ list | length }}               Count items\n'
    + '{{ list | join(\", \") }}           Join list to string\n'
    + '{{ html  | safe }}                Render raw HTML (trusted only!)\n'
    + '{{ input | escape }}              HTML-escape (default)\n'
    + '{{ num   | round }}               Round a number\n'
    + '{{ str   | int }}                 Convert to int\n\n'
    + '3. TEMPLATE INHERITANCE\n'
    + '-' * 58 + '\n'
    + '# base.html\n'
    + '{% block title %}Default{% endblock %}\n'
    + '{% block content %}{% endblock %}\n'
    + '{% block scripts %}{% endblock %}\n\n'
    + '# child.html\n'
    + '{% extends \"base.html\" %}\n'
    + '{% block title %}My Page{% endblock %}\n'
    + '{% block content %}\n'
    + '  <h1>My unique content</h1>\n'
    + '{% endblock %}\n\n'
    + '# Call parent block content with super()\n'
    + '{% block content %}\n'
    + '  {{ super() }}\n'
    + '  <p>Extra content after parent.</p>\n'
    + '{% endblock %}\n\n'
    + '4. RENDER_TEMPLATE\n'
    + '-' * 58 + '\n'
    + 'from flask import render_template, render_template_string\n\n'
    + 'return render_template(\"page.html\", key=value, items=lst)\n'
    + 'return render_template_string(\"<h1>{{ title }}</h1>\", title=\"Hi\")\n\n'
    + '5. STATIC FILES\n'
    + '-' * 58 + '\n'
    + '{{ url_for(\"static\", filename=\"css/style.css\") }}\n'
    + '{{ url_for(\"static\", filename=\"js/app.js\") }}\n'
    + '{{ url_for(\"static\", filename=\"img/logo.png\") }}\n\n'
    + '6. MACROS\n'
    + '-' * 58 + '\n'
    + '{% macro button(text, type=\"button\", cls=\"btn\") %}\n'
    + '  <button type="{{ type }}" class="{{ cls }}">{{ text }}</button>\n'
    + '{% endmacro %}\n'
    + '{{ button(\"Save\", type=\"submit\", cls=\"btn btn-primary\") }}\n\n'
    + '7. LOOP VARIABLES\n'
    + '-' * 58 + '\n'
    + 'loop.index    -> 1-based position\n'
    + 'loop.index0   -> 0-based position\n'
    + 'loop.first    -> True on first iteration\n'
    + 'loop.last     -> True on last iteration\n'
    + 'loop.length   -> total items\n'
    + '=' * 58
)
print(cheat_sheet)

---

## 💪 Practice Prompts

Work through these challenges to solidify your Jinja2 skills.

---

### Challenge 1: Build a full site skeleton with template inheritance

**Goal:** Create a `base.html` with a navbar and footer, then create 3 child pages.

**Requirements:**
- `base.html` must have: `{% block title %}`, `{% block content %}`, `{% block scripts %}`
- Navbar links: Home, Blog, Contact
- `home.html` — welcome message + list of 3 recent posts
- `blog.html` — loop rendering 5 blog post titles
- `contact.html` — a simple contact form (HTML only)

**Stretch:** Add `{% block active_page %}{% endblock %}` to highlight the active nav link.

---

### Challenge 2: Build a reusable card macro

**Goal:** Create a `card` macro and use it 3 times with different data.

**Requirements:**
- Signature: `card(title, body, footer=None, color='default')`
- Render `<div class="card card-{{ color }}">`
- Show `<div class="card-footer">` only when `footer` is provided
- Use for: a product card, a user profile card, an article preview

**Stretch:** Create an `icon_badge(icon, count)` macro and nest it inside the product card.

---

### Challenge 3: Blog post list with a "New!" badge

**Goal:** Render a list of blog posts and show a "🆕 New!" badge for posts published within the last 7 days.

**Requirements:**
- Use `from datetime import datetime, timedelta` for sample posts
- Each post: `title`, `author`, `published_at` (datetime), `excerpt`
- Show "🆕 New!" if `published_at > datetime.now() - timedelta(days=7)`
- Use `loop.index` to number posts
- Use `| default` for optional fields

**Stretch:** Show "Showing X of Y posts" and an empty-state message when the list is empty.


In [ ]:
from datetime import datetime, timedelta
from jinja2 import Environment

env = Environment()
now = datetime.now()

posts = [
    {"title": "Getting Started with Flask",       "author": "Alice",   "published_at": now - timedelta(days=1),  "excerpt": "Flask is a lightweight Python web framework..."},
    {"title": "Jinja2 Template Inheritance",       "author": "Bob",     "published_at": now - timedelta(days=5),  "excerpt": "Template inheritance lets you build a base layout..."},
    {"title": "Working with Flask-SQLAlchemy",     "author": "Alice",   "published_at": now - timedelta(days=10), "excerpt": "SQLAlchemy makes database work much easier..."},
    {"title": "Deploying Flask to Production",     "author": "Charlie", "published_at": now - timedelta(days=3),  "excerpt": "Production deployment requires careful configuration..."},
    {"title": "Flask Blueprints for Large Apps",   "author": "Bob",     "published_at": now - timedelta(days=20), "excerpt": "Blueprints help organize large Flask applications..."},
]

cutoff = now - timedelta(days=7)

template_str = (
    "Blog Posts ({{ posts | length }} total)\n"
    "========================================\n"
    "{%- for post in posts %}\n"
    "{{ loop.index }}. {{ post.title }}\n"
    "   By {{ post.author | default('Anonymous') }}  |  "
    "{{ post.published_at.strftime('%b %d, %Y') }}\n"
    "   {% if post.published_at > cutoff %}NEW!{% endif %}\n"
    "   {{ post.excerpt | truncate(60) }}\n"
    "{% endfor %}"
)

result = env.from_string(template_str).render(posts=posts, cutoff=cutoff)
print(result)


---

## 🎓 Key Takeaways

| Concept | One-liner |
|---------|-----------|
| **Template engine** | Fills dynamic slots in HTML with Python data |
| **Jinja2 syntax** | `{{ output }}`, `{% control %}`, `{# comment #}` |
| **Filters** | Transform values inline: `\| upper`, `\| default('N/A')`, `\| truncate(n)` |
| **render_template()** | Flask function that loads + renders a `.html` file from `templates/` |
| **Template inheritance** | `base.html` once → all child pages update; child uses `{% extends %}` |
| **Macros** | Reusable HTML components — like Python functions for markup |
| **Static files** | Always `url_for('static', filename='...')`, never hardcode |
| **Auto-escaping** | Jinja2 escapes `<>` by default — built-in XSS protection |
| **`\| safe`** | Disables escaping — only use on server-generated HTML, never user input |

**You're now ready to build real Flask pages with proper structure, reusable layouts, and dynamic data!**


---

## 🏆 Putting It All Together: A Mini Blog Preview

Let's combine everything — inheritance, loops, filters, conditionals, and macros — in one cohesive demo.


In [ ]:
from jinja2 import Environment, DictLoader
from datetime import datetime, timedelta

now = datetime.now()

BASE = (
    "<!DOCTYPE html>\n<html>\n<head>\n"
    "  <title>{% block title %}Flask Blog{% endblock %}</title>\n"
    "</head>\n<body>\n"
    "<nav>\n"
    "{%- for link in nav_links %}\n"
    "  <a href=\"{{ link.url }}\"{% if link.active %} style=\"font-weight:bold\"{% endif %}>{{ link.label }}</a>"
    "{%- if not loop.last %} |{% endif %}\n"
    "{%- endfor %}\n"
    "</nav>\n<hr>\n<main>\n"
    "{% block content %}{% endblock %}\n"
    "</main>\n<hr>\n"
    "<footer>© 2024 Flask Blog -- Built with Jinja2</footer>\n"
    "</body>\n</html>"
)

POST_LIST = (
    '{% extends "base.html" %}\n'
    "{% block title %}All Posts -- Flask Blog{% endblock %}\n"
    "{% block content %}\n"
    "{%- macro post_card(post) -%}\n"
    "  [{{ loop.index }}] {{ post.title }}\n"
    "       By {{ post.author | default('Anonymous') }} | {{ post.date }}\n"
    "       {{ post.excerpt | truncate(70) }}\n"
    "       {% if post.is_new %}NEW!{% endif %}\n"
    "{%- endmacro -%}\n\n"
    "<h1>Latest Posts ({{ posts | length }})</h1>\n"
    "{% for post in posts %}\n"
    "{{ post_card(post) }}\n"
    "{% else %}\n"
    "  No posts yet. Check back soon!\n"
    "{% endfor %}\n"
    "{% endblock %}"
)

env = Environment(loader=DictLoader({"base.html": BASE, "post_list.html": POST_LIST}),
                  trim_blocks=True, lstrip_blocks=True)

nav = [
    {"label": "Home",  "url": "/",     "active": False},
    {"label": "Blog",  "url": "/blog", "active": True},
    {"label": "About", "url": "/about","active": False},
]

posts_data = [
    {"title": "Getting Started with Flask",    "author": "Alice",   "date": (now - timedelta(days=2)).strftime("%b %d"), "excerpt": "Flask is a micro web framework written in Python.",               "is_new": True},
    {"title": "Jinja2 Template Magic",          "author": "Bob",     "date": (now - timedelta(days=6)).strftime("%b %d"), "excerpt": "Templates separate HTML from Python business logic.",            "is_new": True},
    {"title": "Flask-SQLAlchemy Deep Dive",     "author": "Alice",   "date": (now - timedelta(days=15)).strftime("%b %d"), "excerpt": "SQLAlchemy integrates beautifully with Flask apps.",           "is_new": False},
    {"title": "Deploying to Production",        "author": None,      "date": (now - timedelta(days=30)).strftime("%b %d"), "excerpt": "Going live requires careful server and database setup.",       "is_new": False},
]

print(env.get_template("post_list.html").render(nav_links=nav, posts=posts_data))


---

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='2. routing_and_url_building.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 2: Routing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='../2.%20working_with_data/4.%20handling_forms_and_user_input.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 4: Forms →</a>
</div>
